# Part 1 — EDA & Audit
Snapshot date: 2025-09-30. Target: churn_next_60d.

This notebook loads all raw datasets, runs the audit, and produces the charts saved under `charts/`.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
D='../data'
customers=pd.read_csv(f'{D}/customers.csv', parse_dates=['signup_date'])
orders=pd.read_csv(f'{D}/orders.csv', parse_dates=['order_date'])
tickets=pd.read_csv(f'{D}/support_tickets.csv', parse_dates=['ticket_date'])
labels=pd.read_csv(f'{D}/churn_labels.csv')
intervention=pd.read_csv(f'{D}/intervention_history.csv')
rfm=pd.read_csv(f'{D}/rfm_modeling_snapshot.csv')
print(customers.shape, orders.shape, tickets.shape, labels.shape, intervention.shape, rfm.shape)


## Schema & null audit

In [ ]:
for name, df in [('customers',customers),('orders',orders),('tickets',tickets),('labels',labels),('intervention',intervention)]:
    print(name); print(df.isna().sum()); print('---')


## Leakage check — orders dated after snapshot

In [ ]:
SNAP=pd.Timestamp('2025-09-30')
print('post-snapshot orders:', (orders.order_date>SNAP).sum())
orders_pre = orders[orders.order_date<=SNAP].copy()


## Churn distribution and segment cuts (charts saved under charts/)

In [ ]:
print(labels.churn_next_60d.value_counts(normalize=True))
m = customers.merge(labels, on='customer_id')
print(m.groupby('acquisition_channel').churn_next_60d.mean().sort_values())
print(m.assign(lt=m.loyalty_tier.fillna('None')).groupby('lt').churn_next_60d.mean())


## Five Churn-Risk Hypotheses (evidence in charts/)
1. **Inactivity hypothesis** — customers with `sessions_30d == 0` churn at very high rates (chart 05).
2. **Recency hypothesis** — churn rate climbs monotonically with `recency_days` (chart 03).
3. **Support pain hypothesis** — `ticket_count_90d >= 2` correlates with higher churn (chart 04).
4. **Channel hypothesis** — Marketplace/Google Search cohorts churn faster than Organic/Referral (chart 02).
5. **Loyalty hypothesis** — non-enrolled customers churn at roughly 2× the rate of Gold/Platinum (chart 06).